# AST (Audio Spectrogram Transformer)


## 1. Setup

In [1]:
import numpy as np
import torch
import librosa

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

from transformers import ASTFeatureExtractor, ASTModel

from kitty3000_ml.preprocess import load_clip
from kitty3000_ml.labels import LABELS             # MK: import the label set for classification from central source

/home/eva20/code/Misakis204/Kitty3000-ML/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = "../data/raw/CatSound_originals"                   # MK: unzip the originals file from  gdrive into the data directory
MANIFEST = "../data/manifest_v2_clean.csv"                             # MK: or ../data/manifest_dirty.csv

SAMPLE_RATE = 16000                                           # AST's expected input rate
SECONDS = 11.0                                                # MK: team decision 09.09., based on Ceven's duration tests (2/5/7/11 s)

CLASS_NAMES = [l for l in LABELS if l != "Unknown"]   # 10 classes, same order as the API (labels.py); Unknown is V2

MODEL_NAME = "ast_audioset_frozen_logreg"
AST_CHECKPOINT = "MIT/ast-finetuned-audioset-10-10-0.4593"

## 2. Load dataset


In [8]:
manifest = pd.read_csv(MANIFEST)

manifest = pd.read_csv(MANIFEST)

def load_split(split):
    rows = manifest[manifest["split"] == split]
    waveforms = []
    labels = []
    skipped = []
    for i in rows.index:
        path = DATA_DIR + "/" + rows["path"][i]
        try:
            waveforms.append(load_clip(path, SAMPLE_RATE, SECONDS))
            labels.append(CLASS_NAMES.index(rows["label"][i]))  # "Defence" -> 1, same order as labels.py
        except Exception as e:
            skipped.append((path, str(e)))
    if skipped:
        print(f"Skipped {len(skipped)} unreadable files in '{split}':")
        for p, err in skipped[:10]:
            print(f"  {p}: {err}")
    return waveforms, labels

train_waveforms, train_labels = load_split("train")
test_waveforms, test_labels = load_split("val")     # val for now - test stays untouched

print(f"\nLoaded {len(train_waveforms) + len(test_waveforms)} clips across {len(CLASS_NAMES)} classes")

Skipped 2041 unreadable files in 'train':
  ../data/raw/CatSound_originals/Angry/Cat_Angry1026_aug1(1).mp3: Error opening '../data/raw/CatSound_originals/Angry/Cat_Angry1026_aug1(1).mp3': System error.
  ../data/raw/CatSound_originals/Angry/Cat_Angry1027_aug1(1).mp3: Error opening '../data/raw/CatSound_originals/Angry/Cat_Angry1027_aug1(1).mp3': System error.
  ../data/raw/CatSound_originals/Angry/Cat_Angry1029_aug1(1).mp3: Error opening '../data/raw/CatSound_originals/Angry/Cat_Angry1029_aug1(1).mp3': System error.
  ../data/raw/CatSound_originals/Angry/Cat_Angry1032_aug1(1).mp3: Error opening '../data/raw/CatSound_originals/Angry/Cat_Angry1032_aug1(1).mp3': System error.
  ../data/raw/CatSound_originals/Angry/Cat_Angry1033_aug1(1).mp3: Error opening '../data/raw/CatSound_originals/Angry/Cat_Angry1033_aug1(1).mp3': System error.
  ../data/raw/CatSound_originals/Angry/Cat_Angry1034_aug1(1).mp3: Error opening '../data/raw/CatSound_originals/Angry/Cat_Angry1034_aug1(1).mp3': System error

Note: Illegal Audio-MPEG-Header 0xbf082800 at offset 7536.
Note: Trying to resync...
Note: Hit end of (available) data during resync.



Loaded 2506 clips across 10 classes


In [9]:
print(f"Train: {len(train_waveforms)} | Val: {len(test_waveforms)}")

Train: 2058 | Val: 448


## 3. Load AST and extract embeddings


In [10]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

feature_extractor = ASTFeatureExtractor.from_pretrained(AST_CHECKPOINT)
ast_model = ASTModel.from_pretrained(AST_CHECKPOINT).to(device)
ast_model.eval()


def extract_ast_embedding(waveform, sr=SAMPLE_RATE):
    """Returns a single fixed-length embedding vector for one clip (mean-pooled over patches)."""
    inputs = feature_extractor(waveform, sampling_rate=sr, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = ast_model(**inputs)

    last_hidden = outputs.last_hidden_state
    pooled = last_hidden.mean(dim=1).squeeze(0)
    return pooled.cpu().numpy()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4439.93it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
print("Extracting embeddings for train set...")
X_train = np.stack([extract_ast_embedding(w) for w in train_waveforms])

print("Extracting embeddings for test set...")
X_test = np.stack([extract_ast_embedding(w) for w in test_waveforms])

print(X_train.shape, X_test.shape)

Extracting embeddings for train set...
Extracting embeddings for test set...
(2058, 768) (448, 768)


In [12]:
np.savez(f"ast_{SECONDS}s.npz",
    X_train=X_train, y_train=np.array(train_labels),
    X_test=X_test, y_test=np.array(test_labels),
    sr=SAMPLE_RATE, seconds=SECONDS)

## 4. Train classifier head


In [13]:
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(X_train, train_labels)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

## 5. Evaluate

In [14]:
def ast_predict_fn(waveform, sr):
    embedding = extract_ast_embedding(waveform, sr)
    return clf.predict(embedding[None, :])[0]


y_true = test_labels
y_pred = clf.predict(X_test)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Macro F1: {macro_f1:.4f}")

              precision    recall  f1-score   support

       Angry       0.91      0.91      0.91        45
     Defence       0.93      0.98      0.96        44
    Fighting       0.88      0.96      0.91        45
       Happy       0.79      0.91      0.85        45
 HuntingMind       0.98      0.93      0.95        44
      Mating       0.84      0.82      0.83        45
  MotherCall       0.93      0.89      0.91        46
     Paining       0.84      0.73      0.78        44
     Resting       1.00      1.00      1.00        45
     Warning       0.81      0.78      0.80        45

    accuracy                           0.89       448
   macro avg       0.89      0.89      0.89       448
weighted avg       0.89      0.89      0.89       448

Macro F1: 0.8899


## 6. Compare against PANN

(PANN, AST, and any future models)

In [ ]:
#Not ready yet

## 7. Duration-bias sanity check

In [15]:
val_rows = manifest[manifest["split"] == "val"]
durations = [librosa.get_duration(path=DATA_DIR + "/" + p) for p in val_rows["path"]]
misclassified = [
    (round(d, 1), CLASS_NAMES[t], CLASS_NAMES[p])
    for d, t, p in zip(durations, y_true, y_pred)
    if t != p
]
print(f"{len(misclassified)} misclassified clips — inspect for duration patterns:")
misclassified[:10]

49 misclassified clips — inspect for duration patterns:


[(6.6, 'Angry', 'Fighting'),
 (3.5, 'Angry', 'Paining'),
 (3.0, 'Angry', 'Paining'),
 (2.6, 'Angry', 'Warning'),
 (4.4, 'Defence', 'Warning'),
 (1.4, 'Fighting', 'Happy'),
 (2.6, 'Fighting', 'Warning'),
 (1.4, 'Happy', 'Paining'),
 (4.4, 'Happy', 'Mating'),
 (2.7, 'Happy', 'Paining')]

In [16]:
import joblib

joblib.dump({
    "model": clf,
    "classes": CLASS_NAMES,
    "sr": SAMPLE_RATE,
    "seconds": SECONDS,
    "checkpoint": AST_CHECKPOINT,
}, "ast_logreg.joblib")

['ast_logreg.joblib']